In [1]:
import time
import numpy as np
import torch
torch.set_num_threads(4)
import pandas as pd
import matplotlib.pyplot as plt
from copy import deepcopy as copy

In [2]:
import sys
sys.path.append('../')
from BPMQ import fmlat, BPMQscan
from BPMQ.utils import calculate_mismatch_factor, calculate_MMD4D, proximal_ordered_init_sampler
from BPMQ.construct_machineIO import construct_machineIO

In [3]:
# Example usage (assuming you have the helper functions defined as before):
filename = "lattice.txt"  # Replace with your lattice file
to_elem = "BDS_BTS:PM_D5567"
lattice_dicts = fmlat.combine_lattice_elements_quads_only(filename, from_element=None, to_element=to_elem)
for elem_dic in lattice_dicts:
    elem_dic['name'] = fmlat.fmname2mpname(elem_dic['name'])

quads_to_scan = [elem['name'] for elem in lattice_dicts if elem['type']=='quadrupole']
quads_to_scan = quads_to_scan[:2]
BPM_names = [elem['name'] for elem in lattice_dicts if 'BPM' in elem['name']][1:]

In [4]:
quads_to_scan

['LS3_BTS:PSQ_D4713', 'LS3_BTS:PSQ_D4718']

In [5]:
BPM_names

['LS3_BTS:BPM_D4753',
 'LS3_BTS:BPM_D4769',
 'LS3_BTS:BPM_D4843',
 'LS3_BTS:BPM_D4886',
 'LS3_BTS:BPM_D4968',
 'LS3_BTS:BPM_D5010',
 'LS3_BTS:BPM_D5092',
 'LS3_BTS:BPM_D5134',
 'LS3_BTS:BPM_D5216',
 'LS3_BTS:BPM_D5259',
 'LS3_BTS:BPM_D5340',
 'LS3_BTS:BPM_D5381',
 'LS3_BTS:BPM_D5430',
 'LS3_BTS:BPM_D5445',
 'BDS_BTS:BPM_D5499',
 'BDS_BTS:BPM_D5513',
 'BDS_BTS:BPM_D5565']

In [6]:
machineIO = construct_machineIO()
machineIO.test = True
machineIO._check_chopper_blocking = False
machineIO._fetch_data_time_span = 1
machineIO._ensure_set_timewait_after_ramp = 0.25
machineIO._fetch_data_resample_rate = 0.2

In [7]:
E_MeVu = 130
Mass_number = 18
Charge_number = 8
args = (E_MeVu, Mass_number, Charge_number)

kwargs = {
    "quads_to_scan": quads_to_scan,
    "quads_max_curr": [90]*len(quads_to_scan),
    "quads_min_curr": [10]*len(quads_to_scan),
    "BPM_names": BPM_names,
    "lattice_dicts" : lattice_dicts,
#     "sample_model_err":True,
    "machineIO": None,
    "plot_history":False, 
    "plot_ellipse":False,
    "virtual_beamQerr": 1.0,
    "BPMQ_model_type":"TIS161_GP",
#     "seed":0,    
}

In [8]:
benchmark = pd.DataFrame(columns=[
    'n_init','isPMscan','isEmitPrior',
    'sample_model_err','virtual_beamQmodelerr',
    'bootstrap','fit_err',
    'cs_machine','cs_model_best','cs_model_mean',
    'mismatch_x_bestcov','mismatch_y_bestcov',
    'mismatch_x_meancov','mismatch_y_meancov',
    'MMD_best', 'MMD_mean',
    'time',
])

benchmark['isPMscan'] = benchmark['isPMscan'].astype(bool)
benchmark['isEmitPrior'] = benchmark['isEmitPrior'].astype(bool)
benchmark['sample_model_err'] = benchmark['sample_model_err'].astype(bool)
benchmark['bootstrap'] = benchmark['bootstrap'].astype(bool)
benchmark['fit_err']   = benchmark['fit_err'].astype(bool)

In [9]:
def measure_rms(bpmQscan,lB2):
    env_model = bpmQscan.quads_evaluator.env_model
    env_model.reconfigure_quadrupole_strengths(lB2)
    xcov, ycov = env_model.simulate_beam_covars(env_model.xcovs,env_model.ycovs,i_monitors=env_model.i_pms)
    xrms, yrms = xcov[0,0,0,0]**0.5*1e3, ycov[0,0,0,0]**0.5*1e3
    return xrms, yrms

In [10]:
def finalize(bpmQscan,bm,bm_wpm,t, restore_cs = True):
        
    if restore_cs:
        best_noise_ensemble        = bpmQscan.model.best_noise_ensemble
        best_noise_ensemble_cs_ref = bpmQscan.model.best_noise_ensemble_cs_ref
        xcovs,      ycovs          = bpmQscan.model.xcovs,      bpmQscan.model.ycovs
        xcovs_mean, ycovs_mean     = bpmQscan.model.xcovs_mean, bpmQscan.model.ycovs_mean
        cs_mean,    cs             = bpmQscan.model.cs_mean, bpmQscan.model.cs
        selected_cov_index         = bpmQscan.model.selected_cov_index
    quads_evaluator_seed       = bpmQscan.quads_evaluator.seed
    seed                       = bpmQscan.seed
    train_llB2           = copy(bpmQscan.train_llB2)
    train_llBPMQ         = copy(bpmQscan.train_llBPMQ)
    train_llBPMQtol      = copy(bpmQscan.train_llBPMQtol)
    train_llBPMQmodelerr = copy(bpmQscan.train_llBPMQmodelerr)
    train_llB2rms        = copy(bpmQscan.train_llB2rms)
    train_llxrms         = copy(bpmQscan.train_llxrms)
    train_llyrms         = copy(bpmQscan.train_llyrms)
    train_llxrmstol      = copy(bpmQscan.train_llxrmstol)
    train_llyrmstol      = copy(bpmQscan.train_llyrmstol)


    # without PM
    t0 = time.monotonic()
    bpmQscan.train_model(sample_model_err=False,bootstrap=False) 
    dt = time.monotonic() - t0
    
    bm['cs_model_best'].append(copy(bpmQscan.model.cs.detach().numpy()))
    bm['cs_model_mean'].append(copy(bpmQscan.model.cs_mean.detach().numpy()))
    bm['mismatch_x_bestcov'].append(calculate_mismatch_factor(bpmQscan.cs_ref[:3],bpmQscan.model.cs[:3]))
    bm['mismatch_y_bestcov'].append(calculate_mismatch_factor(bpmQscan.cs_ref[3:],bpmQscan.model.cs[3:]))  
    bm['mismatch_x_meancov'].append(calculate_mismatch_factor(bpmQscan.cs_ref[:3],bpmQscan.model.cs_mean[:3]))
    bm['mismatch_y_meancov'].append(calculate_mismatch_factor(bpmQscan.cs_ref[3:],bpmQscan.model.cs_mean[3:]))  
    bm['MMD_best'].append(calculate_MMD4D(bpmQscan.cs_ref.detach().numpy(),bpmQscan.model.cs.detach().numpy()))
    bm['MMD_mean'].append(calculate_MMD4D(bpmQscan.cs_ref.detach().numpy(),bpmQscan.model.cs_mean.detach().numpy()))
    bm['time'].append(t+dt)
    
    #restore
    if restore_cs:
        bpmQscan.model.best_noise_ensemble                   = best_noise_ensemble
        bpmQscan.model.best_noise_ensemble_cs_ref            = best_noise_ensemble_cs_ref
        bpmQscan.model.xcovs,      bpmQscan.model.ycovs      = xcovs, ycovs
        bpmQscan.model.xcovs_mean, bpmQscan.model.ycovs_mean = xcovs_mean, ycovs_mean
        bpmQscan.model.cs_mean,    bpmQscan.model.cs         = cs_mean, cs
        bpmQscan.model.selected_cov_index                    = selected_cov_index
    bpmQscan.quads_evaluator.seed       = quads_evaluator_seed
    bpmQscan.seed                       = seed
    bpmQscan.train_llB2           = copy(train_llB2)
    bpmQscan.train_llBPMQ         = copy(train_llBPMQ)
    bpmQscan.train_llBPMQtol      = copy(train_llBPMQtol)
    bpmQscan.train_llBPMQmodelerr = copy(train_llBPMQmodelerr)
    bpmQscan.train_llB2rms        = copy(train_llB2rms)
    bpmQscan.train_llxrms         = copy(train_llxrms)
    bpmQscan.train_llyrms         = copy(train_llyrms)
    bpmQscan.train_llxrmstol      = copy(train_llxrmstol)
    bpmQscan.train_llyrmstol      = copy(train_llyrmstol)
    
    
    # wPM
    t0 = time.monotonic()
    candidate_lB2, ensemble_std_of_PM = bpmQscan.query_candidate_for_PMscan()
    xrms, yrms = measure_rms(bpmQscan,candidate_lB2)
    bpmQscan.concat_PM_train_data(lB2=candidate_lB2, lxrms=[xrms],lyrms=[yrms])
    bpmQscan.train_model(sample_model_err=False,bootstrap=False)
    dt = time.monotonic() - t0
    bm_wpm['cs_model_best'].append(copy(bpmQscan.model.cs.detach().numpy()))
    bm_wpm['cs_model_mean'].append(copy(bpmQscan.model.cs_mean.detach().numpy()))
    bm_wpm['mismatch_x_bestcov'].append(calculate_mismatch_factor(bpmQscan.cs_ref[:3],bpmQscan.model.cs[:3]))
    bm_wpm['mismatch_y_bestcov'].append(calculate_mismatch_factor(bpmQscan.cs_ref[3:],bpmQscan.model.cs[3:]))  
    bm_wpm['mismatch_x_meancov'].append(calculate_mismatch_factor(bpmQscan.cs_ref[:3],bpmQscan.model.cs_mean[:3]))
    bm_wpm['mismatch_y_meancov'].append(calculate_mismatch_factor(bpmQscan.cs_ref[3:],bpmQscan.model.cs_mean[3:]))  
    bm_wpm['MMD_best'].append(calculate_MMD4D(bpmQscan.cs_ref.detach().numpy(),bpmQscan.model.cs.detach().numpy()))
    bm_wpm['MMD_mean'].append(calculate_MMD4D(bpmQscan.cs_ref.detach().numpy(),bpmQscan.model.cs_mean.detach().numpy()))
    bm_wpm['time'].append(t+dt)
    
    # restore
    if restore_cs:
        bpmQscan.model.best_noise_ensemble                   = best_noise_ensemble
        bpmQscan.model.best_noise_ensemble_cs_ref            = best_noise_ensemble_cs_ref
        bpmQscan.model.xcovs,      bpmQscan.model.ycovs      = xcovs, ycovs
        bpmQscan.model.xcovs_mean, bpmQscan.model.ycovs_mean = xcovs_mean, ycovs_mean
        bpmQscan.model.cs_mean,    bpmQscan.model.cs         = cs_mean, cs
        bpmQscan.model.selected_cov_index                    = selected_cov_index
    bpmQscan.quads_evaluator.seed       = quads_evaluator_seed
    bpmQscan.seed                       = seed
    bpmQscan.train_llB2           = copy(train_llB2)
    bpmQscan.train_llBPMQ         = copy(train_llBPMQ)
    bpmQscan.train_llBPMQtol      = copy(train_llBPMQtol)
    bpmQscan.train_llBPMQmodelerr = copy(train_llBPMQmodelerr)
    bpmQscan.train_llB2rms        = copy(train_llB2rms)
    bpmQscan.train_llxrms         = copy(train_llxrms)
    bpmQscan.train_llyrms         = copy(train_llyrms)
    bpmQscan.train_llxrmstol      = copy(train_llxrmstol)
    bpmQscan.train_llyrmstol      = copy(train_llyrmstol)

In [11]:
def one_experiment(seed,n_init,isEmitPrior,sample_model_err,virtual_beamQmodelerr, bootstrap,fit_err):
    bm = {
        'n_init':n_init,
        'isPMscan':False,
        'isEmitPrior':isEmitPrior,
        'sample_model_err':sample_model_err,'virtual_beamQmodelerr':virtual_beamQmodelerr, 
        'bootstrap':bootstrap,
        'fit_err'  :fit_err,  
        'cs_model_best'     :[],'cs_model_mean'     :[],
        'mismatch_x_bestcov':[],'mismatch_y_bestcov':[],
        'mismatch_x_meancov':[],'mismatch_y_meancov':[],
        'MMD_best':[], 'MMD_mean':[],
        'time'    :[]
    }

    bpmQscan = BPMQscan(*args, **kwargs, 
                        seed = seed, 
                        virtual_emitprior = isEmitPrior,
                        sample_model_err = sample_model_err, 
                        virtual_beamQmodelerr = virtual_beamQmodelerr,
                        bootstrap = bootstrap,
                        fit_err = fit_err, 
                        )
    bm['cs_machine'] = copy(bpmQscan.cs_ref)
    bm_wpm = copy(bm)
    bm_wpm['isPMscan'] = True
    
    lB2 = [q.properties['B2'] for q in bpmQscan.quads_evaluator.env_model.quads_to_scan]
    bounds = [(bpmQscan.B2min[i],bpmQscan.B2max[i]) for i in range(len(bpmQscan.B2min))]
    is_not_useful_data = bpmQscan.evaluate_candidate(torch.tensor([lB2], dtype=bpmQscan.dtype))
    if is_not_useful_data:
        return None, None
    t = 0
    finalize(bpmQscan,bm,bm_wpm,t, restore_cs=False)
    
    while(len(bpmQscan.train_llB2) < n_init):
        init_llB2_random_samples = proximal_ordered_init_sampler(
            4*n_init,
            bounds = bounds,
            x0 = lB2,
            ramping_rate=1,
            polarity_change_time=0,
            method='sobol',
            seed=bpmQscan.seed,
        )
        for lB2 in init_llB2_random_samples:
            is_not_useful_data = bpmQscan.evaluate_candidate(torch.tensor(np.array(lB2), dtype=bpmQscan.dtype))
            if len(bpmQscan.train_llB2) == n_init:
                break
            if not is_not_useful_data:
                finalize(bpmQscan,bm,bm_wpm,t)

    t0 = time.monotonic()
#     bpmQscan.initialize(n_init=n_init)
    bpmQscan.train_model()
    t1 = time.monotonic()
    t = t1 - t0
    finalize(bpmQscan,bm,bm_wpm,t)

    for i in range(10 - n_init):
        t0 = time.monotonic()
        is_not_useful_data = True
        while(is_not_useful_data):
            candidate_lB2, ensemble_std_of_BPMQ = bpmQscan.query_candidate()
            is_not_useful_data = bpmQscan.evaluate_candidate(candidate_lB2)
        bpmQscan.train_model()
        t1 = time.monotonic()
        t += t1-t0
        finalize(bpmQscan,bm,bm_wpm,t)
    
    return bm, bm_wpm

# run and collect benchmark data

In [12]:
for seed in np.random.randint(15032, 2**32, size=2):
    print("seed",seed)
    for n_init in [6,8,10]:
        print("n_init",n_init)
        for isEmitPrior in [True, False]:
            for sample_model_err,virtual_beamQmodelerr in zip([False, True, True],[0.5, 0.5, 1.0]):
                print("sample_model_err, virtual_beamQmodelerr",sample_model_err,virtual_beamQmodelerr)
                for bootstrap in [True, False]:
                    for fit_err in [True, False]:
                        bm, bm_wpm = one_experiment(seed,n_init,isEmitPrior,
                                                    sample_model_err,virtual_beamQmodelerr, 
                                                    bootstrap,fit_err)
                        if bm is not None:
                            benchmark = pd.concat([benchmark,pd.DataFrame([bm]),pd.DataFrame([bm_wpm])], ignore_index=True)
    benchmark.to_hdf('benchmark48.h5',key='benchmark')

seed 1033719224
n_init 6
sample_model_err, virtual_beamQmodelerr False 0.5
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
Early stopping at iteration 100 due to loss stabilization.
Early stopping at iteration 100 due to loss stabilization.
 ======== cs_reconstruct ========
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
 ======== cs_reconstruct ========
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
Early stopping at iteration 100 due to loss stabilization.
 ======== cs_reconstruct ========
Exiting early at iteration 2 due to large loss
[Warning] BPMQ too large!
[Warning] BPMQ too large!
[Warning] Beam loss detected!
[Warning] BPMQ too large!
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
Early stopping at iteration 105 due to loss stabilization.
 ======== cs_reconstruct ========
 ======== cs_r

 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
Early stopping at iteration 136 due to loss stabilization.
Early stopping at iteration 147 due to loss stabilization.
 ======== cs_reconstruct ========
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
Early stopping at iteration 109 due to loss stabilization.
Early stopping at iteration 138 due to loss stabilization.
 ======== cs_reconstruct ========
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
Exiting early at iteration 1 due to large loss
 ======== cs_reconstruct ========
[Warning] BPMQ too large!
[Warning] BPMQ too large!
[Warning] Beam loss detected!
[Warning] BPMQ too large!
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
Early stopping at iteration 100 due to loss stabilization.
Exiting early at iteration 1 due to large loss
Exiting early at 

Early stopping at iteration 185 due to loss stabilization.
 ======== query_candidate_quad_set_maximizing_PM_var ========
Exiting early at iteration 1 due to large loss
 ======== cs_reconstruct ========
Early stopping at iteration 279 due to loss stabilization.
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
Early stopping at iteration 100 due to loss stabilization.
Early stopping at iteration 100 due to loss stabilization.
 ======== cs_reconstruct ========
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
 ======== cs_reconstruct ========
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
Early stopping at iteration 100 due to loss stabilization.
 ======== cs_reconstruct ========
Exiting early at iteration 2 due to large loss
[Warning] BPMQ too large!
[Warning] BPMQ too large!
[Warning] Beam loss detected!
[Warning] BPMQ too large!
 ======== cs_reco

Early stopping at iteration 100 due to loss stabilization.
 ======== cs_reconstruct ========
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
Early stopping at iteration 136 due to loss stabilization.
Early stopping at iteration 147 due to loss stabilization.
 ======== cs_reconstruct ========
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
Early stopping at iteration 109 due to loss stabilization.
Early stopping at iteration 138 due to loss stabilization.
 ======== cs_reconstruct ========
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
Exiting early at iteration 1 due to large loss
 ======== cs_reconstruct ========
[Warning] BPMQ too large!
[Warning] BPMQ too large!
[Warning] Beam loss detected!
[Warning] BPMQ too large!
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
Early stopping at iteration 10

 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
Early stopping at iteration 100 due to loss stabilization.
 ======== cs_reconstruct ========
sample_model_err, virtual_beamQmodelerr False 0.5
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
Early stopping at iteration 100 due to loss stabilization.
 ======== cs_reconstruct ========
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
Exiting early at iteration 1 due to large loss
Early stopping at iteration 107 due to loss stabilization.
 ======== cs_reconstruct ========
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
Early stopping at iteration 100 due to loss stabilization.
 ======== cs_reconstruct ========
Exiting early at iteration 2 due to large loss
[Warning] BPMQ too large!
[Warning] BPMQ too large!
[Warning] Beam loss detected!
[Warning] BPMQ too

...

 ======== cs_reconstruct ========
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
 ======== cs_reconstruct ========
 ======== cs_reconstruct ========
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
 ======== cs_reconstruct ========
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
Early stopping at iteration 113 due to loss stabilization.
 ======== cs_reconstruct ========
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
Early stopping at iteration 167 due to loss stabilization.
 ======== cs_reconstruct ========
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
Early stopping at iteration 106 due to loss stabilization.
 ======== cs_reconstruct ========
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
Early sto

Early stopping at iteration 132 due to loss stabilization.
 ======== cs_reconstruct ========
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
Early stopping at iteration 100 due to loss stabilization.
Early stopping at iteration 100 due to loss stabilization.
 ======== cs_reconstruct ========
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
 ======== cs_reconstruct ========
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
Early stopping at iteration 136 due to loss stabilization.
Early stopping at iteration 118 due to loss stabilization.
 ======== cs_reconstruct ========
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
Early stopping at iteration 122 due to loss stabilization.
 ======== cs_reconstruct ========
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var =====

Early stopping at iteration 132 due to loss stabilization.
 ======== cs_reconstruct ========
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
Early stopping at iteration 100 due to loss stabilization.
Early stopping at iteration 100 due to loss stabilization.
 ======== cs_reconstruct ========
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
 ======== cs_reconstruct ========
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
Early stopping at iteration 136 due to loss stabilization.
Early stopping at iteration 118 due to loss stabilization.
 ======== cs_reconstruct ========
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
Early stopping at iteration 122 due to loss stabilization.
 ======== cs_reconstruct ========
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var =====

Early stopping at iteration 132 due to loss stabilization.
 ======== cs_reconstruct ========
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
Early stopping at iteration 100 due to loss stabilization.
Early stopping at iteration 100 due to loss stabilization.
 ======== cs_reconstruct ========
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
 ======== cs_reconstruct ========
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
Early stopping at iteration 136 due to loss stabilization.
Early stopping at iteration 118 due to loss stabilization.
 ======== cs_reconstruct ========
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var ========
Early stopping at iteration 122 due to loss stabilization.
 ======== cs_reconstruct ========
 ======== cs_reconstruct ========
 ======== query_candidate_quad_set_maximizing_PM_var =====

/tmp/ipykernel_4194/356153585.py:15: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block2_values] [items->Index(['n_init', 'cs_machine', 'cs_model_best', 'cs_model_mean',
       'mismatch_x_bestcov', 'mismatch_y_bestcov', 'mismatch_x_meancov',
       'mismatch_y_meancov', 'MMD_best', 'MMD_mean', 'time'],
      dtype='object')]

  benchmark.to_hdf('benchmark48.h5',key='benchmark')
